# Testing the weighted deduplication algorithm on development datasets

This notebook runs the same end-to-end workflow on the **SRSR**, **Cardiac**, and **Neuroimaging** datasets.

The record workflow is object-first:

1. `load_reference_csv` returns validated `Paper` objects.
2. `build_blocked_pairs` creates candidate pairs directly from those objects.
3. `Deduper.score_pair` performs the weighted comparison and returns the probability, field scores, and any early-stop reason.
4. Pandas DataFrames are created for metrics, review tables, and CSV exports.

Because these datasets were involved in model and rule development, the results are an optimistic in-sample check rather than a held-out performance estimate.

## Evaluation strategy

Two complementary views are retained:

1. **Pair level** — sensitivity, specificity, precision, F1, ROC AUC, and average precision across score thresholds.
2. **Record level (ASySD-style)** — candidate pairs are clustered and the resulting kept/removed decisions are compared with the gold-standard duplicate groups.

The record-level view is especially useful operationally: a false positive means a unique paper was discarded, while a false negative means a duplicate survived.


## Setup

The app's current APIs provide the complete record flow:

- `load_reference_csv` parses each row into a `Paper`.
- `build_blocked_pairs` applies the shared blocking rules to `Paper` objects.
- `Deduper.score_pair` owns weighted scoring, field comparisons, the sigmoid transform, DOI penalty, and early-stop rules.
- `ScorePairConfig` makes the intended evaluation weights explicit.

Notebook helpers below are limited to orchestration, metrics, and side-by-side review exports. They do not reproduce deduplication logic.


In [1]:
import sys
from collections.abc import Sequence
from pathlib import Path

import pandas as pd
from loguru import logger
from sklearn.metrics import average_precision_score, confusion_matrix, roc_auc_score
from tqdm.auto import tqdm

repo_root = Path.cwd()
if not (repo_root / "app").exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

results_root = repo_root / "notebooks" / "results"

from algorithm.evaluation import SCORE_CONFIG, run_dataset_pipeline

logger.remove()
logger.add(sys.stderr, level="WARNING")

1

In [2]:
DATASETS = {
    "srsr": repo_root / "notebooks" / "data" / "srsr_data.csv",
    "cardiac": repo_root / "notebooks" / "data" / "cardiac_data.csv",
    "neuroimaging": repo_root / "notebooks" / "data" / "neuroimaging_data.csv",
}

dataset_runs = {
    name: run_dataset_pipeline(name, path)
    for name, path in DATASETS.items()
}

summary_df = pd.DataFrame(
    {
        "dataset": dataset_name,
        "n_records": len(result["papers"]),
        "n_pairs": len(result["pairs_df"]),
        "pair_best_threshold": (
            None if result["best_pair"] is None else result["best_pair"]["threshold"]
        ),
        "pair_best_sensitivity": (
            None if result["best_pair"] is None else result["best_pair"]["sensitivity"]
        ),
        "pair_best_specificity": (
            None if result["best_pair"] is None else result["best_pair"]["specificity"]
        ),
        "record_best_threshold": (
            None if result["best_record"] is None else result["best_record"]["threshold"]
        ),
        "record_best_sensitivity": (
            None if result["best_record"] is None else result["best_record"]["sensitivity"]
        ),
        "record_best_specificity": (
            None if result["best_record"] is None else result["best_record"]["specificity"]
        ),
    }
    for dataset_name, result in dataset_runs.items()
)

summary_df


Scoring candidate pairs:   0%|          | 0/1019402 [00:00<?, ?it/s]

Scoring candidate pairs:   0%|          | 0/62197 [00:00<?, ?it/s]

Scoring candidate pairs:   0%|          | 0/9208 [00:00<?, ?it/s]

,dataset,n_records,n_pairs,pair_best_threshold,pair_best_sensitivity,pair_best_specificity,record_best_threshold,record_best_sensitivity,record_best_specificity
0,srsr,53001,1019402,0.85,0.991407,0.999976,0.85,0.994245,0.999723
1,cardiac,8948,62197,0.85,0.993981,0.999949,0.85,0.994618,0.999446
2,neuroimaging,3438,9208,0.75,0.990066,0.998378,0.75,0.991525,0.997664


In [3]:
MIN_SENSITIVITY = 0.98
THRESHOLD_OVERRIDE = 0.85

if "dataset_runs" not in globals():
    msg = "Run the dataset pipeline cell first."
    raise RuntimeError(msg)

final_record_level_rows = []
for dataset_name, result in dataset_runs.items():
    record_metrics_df = result["record_metrics_df"]

    if THRESHOLD_OVERRIDE is not None:
        matched = record_metrics_df[
            record_metrics_df["threshold"].sub(THRESHOLD_OVERRIDE).abs() < 1e-12
        ]
        if matched.empty:
            msg = (
                f"Threshold {THRESHOLD_OVERRIDE} not found for {dataset_name}. "
                f"Available thresholds: {sorted(record_metrics_df['threshold'].tolist())}"
            )
            raise ValueError(
                msg
            )
        chosen = matched.iloc[0]
        selection_note = "manual_threshold_override"
    else:
        chosen = find_best_threshold(record_metrics_df, MIN_SENSITIVITY)
        if chosen is None:
            chosen = record_metrics_df.loc[record_metrics_df["sensitivity"].idxmax()]
            selection_note = f"fallback_max_sensitivity_below_{MIN_SENSITIVITY:.2f}"
        else:
            selection_note = "max_specificity_with_sensitivity_floor"

    final_record_level_rows.append(
        {
            "dataset": dataset_name,
            "threshold": float(chosen["threshold"]),
            "TP": int(chosen["TP"]),
            "FP": int(chosen["FP"]),
            "TN": int(chosen["TN"]),
            "FN": int(chosen["FN"]),
            "sensitivity": float(chosen["sensitivity"]),
            "specificity": float(chosen["specificity"]),
            "precision": float(chosen["precision"]),
            "selection_rule": selection_note,
        }
    )

final_record_level_df = pd.DataFrame(final_record_level_rows).sort_values(
    ["threshold", "dataset"]
).reset_index(drop=True)
final_record_level_df


,dataset,threshold,TP,FP,TN,FN,sensitivity,specificity,precision,selection_rule
0,cardiac,0.85,3511,3,5415,19,0.994618,0.999446,0.999146,manual_threshold_override
1,neuroimaging,0.85,1284,5,2135,14,0.989214,0.997664,0.996121,manual_threshold_override
2,srsr,0.85,16758,10,36136,97,0.994245,0.999723,0.999404,manual_threshold_override


In [4]:
print("INTERCEPT:", SCORE_CONFIG.intercept)
print("SCORE_FIELDS:", SCORE_CONFIG.fields)
print("SCORE_WEIGHTS:", SCORE_CONFIG.weights)

INTERCEPT: -18.688678
SCORE_FIELDS: ['doi', 'title', 'authors', 'year', 'journal', 'pages', 'issue', 'intercept']
SCORE_WEIGHTS: {'doi': 5.304259, 'title': 11.591445, 'authors': 3.704393, 'year': 2.646572, 'journal': 3.179915, 'pages': 3.79124, 'issue': 1.153251, 'intercept': -18.688678}
